# Used Car Price Prediction: Model Performance and Error Analysis Across Price Ranges

**What this notebook is.** A walkthrough of the finished results. It does no computation of its own — it reads the tables and charts produced by the scripts in `src/`, so nothing here can disagree with the study.

**To regenerate everything from the raw file:**

```
cd src
python run_all.py
```

**Research question.** How accurately can machine learning predict used-car *asking* prices for cars it has never seen, and does that accuracy differ between cheap, mid-range and expensive cars?

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent
TAB, FIG = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures"

def table(name):
    return pd.read_csv(TAB / name)

def chart(name):
    display(Image(filename=str(FIG / name)))

meta = json.loads((TAB / "run_metadata.json").read_text())
best = json.loads((TAB / "best_model.json").read_text())["best"]
print(f"Best model: {best}")
print(f"Training cars: {meta['n_train']:,}   held-back cars: {meta['n_test']:,}")
print(f"Price bands at: ${meta['band_cuts'][0]:,.0f} and ${meta['band_cuts'][1]:,.0f}")

## 1. The data, and what cleaning removed

Source: Zenodo record 10457828 (Craigslist Used Cars and Trucks, CC BY 4.0). The file's MD5 fingerprint matches the published value, so this is provably the archived file. 426,880 adverts, posted 4 April to 5 May 2021.

The target is the **asking price**. The dataset records no sale prices.

In [ ]:
display(table("table1_cleaning_funnel.csv"))
chart("fig01_cleaning_funnel.png")

### The repeat-advert problem

More than half the cars that report a VIN (the car's unique serial number) are advertised more than once — one of them 261 times. Ordinary duplicate removal catches **37 rows out of 426,880**, because every advert carries its own unique reference number.

Left in, the same car would sit in both the training and the test set, and the test score would measure recall rather than prediction. De-duplication therefore runs **before** the split.

In [ ]:
display(table("table2_missingness.csv").head(12))
chart("fig02_missing_data.png")

## 2. What the listings look like

In [ ]:
for name in ["fig03_price_distribution.png", "fig04_price_vs_mileage.png",
             "fig05_price_vs_age.png", "fig06_price_by_manufacturer.png",
             "fig07_price_by_type.png"]:
    chart(name)

## 3. Overall accuracy

- **Average miss (MAE)** — the typical size of the error, in dollars.
- **Weighted miss (RMSE)** — the same, but occasional very bad misses count far more.
- **Variation explained (R-squared)** — 0 means no better than always guessing the average price.
- **Median guess** is not a real model. It predicts one price for every car and sets the floor every model must beat.

In [ ]:
display(table("table3_overall_metrics.csv").round(3))
chart("fig08_model_comparison.png")

## 4. The central result: accuracy by price band

The same errors, measured two ways. A $2,000 miss is 40% wrong on a $5,000 car and 4% wrong on a $50,000 one, so the two measures can rank the bands differently — and that disagreement is the finding.

R-squared is deliberately absent here: slicing by price squeezes the price variation inside each band, which drives R-squared toward zero for every model regardless of quality.

In [ ]:
bands = table("table4_band_metrics.csv")
display(bands[bands.Model == best].round(2))
chart("fig09_mae_by_band_dollars.png")
chart("fig10_error_by_band_percent.png")
chart("fig11_predicted_vs_actual.png")
chart("fig12_residuals_by_band.png")

## 5. What the badly predicted cars have in common

The "large error" line was fixed from training data before any test result was examined, so it could not be chosen to produce a tidier story.

In [ ]:
print(f"A miss above ${meta['large_error_threshold']:,.0f} counts as a large error")
display(table("table5_large_error_profile.csv").round(3))
chart("fig14_large_error_profile.png")

## 6. What the model relies on

Each column is scrambled in turn and the worsening in accuracy recorded (permutation importance). This shows what the model *leans on*, which is a different question from what makes cars hard to predict — and it says nothing about cause.

In [ ]:
display(table("table6_permutation_importance.csv").round(1))
chart("fig13_permutation_importance.png")

## 7. Robustness checks

**Check 1 — what repeat adverts cost.** The model is re-run without removing them, so the same car can appear on both sides of the split. The gap is how much a study that skipped that step would have flattered itself.

**Check 2 — predicting the logarithm of the price.** Prices are lopsided, and taking logarithms evens that out. But it also quietly reduces the penalty for being wrong about expensive cars, which is the very effect under study, so it is reported as a check and not as the headline.

In [ ]:
d = json.loads((TAB / "table7_dedup_effect.json").read_text())
print(f"repeats removed : {d['deduplicated_rows']:,} cars, average miss ${d['deduplicated_mae']:,.0f}")
print(f"repeats left in : {d['repeats_left_rows']:,} cars, average miss ${d['repeats_left_mae']:,.0f}")
print(f"apparent gain   : ${d['apparent_improvement_usd']:,.0f} ({d['apparent_improvement_pct']:.1f}%) of pure illusion")
chart("fig15_dedup_effect.png")
display(table("table8_log_target.csv").round(2))